<a href="https://colab.research.google.com/github/Shreyash-Singh-1/Summer_Internship/blob/main/task3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from IPython.display import display, HTML

url = 'https://raw.githubusercontent.com/Shreyash-Singh-1/Summer_Internship/refs/heads/main/train.csv'
df_raw = pd.read_csv(url)


df_raw['Order Date'] = pd.to_datetime(df_raw['Order Date'], format='%d/%m/%Y', errors='coerce')
df_raw = df_raw.dropna(subset=['Order Date'])


df_monthly = df_raw.groupby(df_raw['Order Date'].dt.to_period('M'))['Sales'].sum().reset_index()


df_monthly['Date'] = df_monthly['Order Date'].dt.to_timestamp()
df_monthly['Sales'] = pd.to_numeric(df_monthly['Sales'], errors='coerce').fillna(0).round(2)


df_monthly['Time_Index'] = np.arange(len(df_monthly))
df_monthly['Month'] = df_monthly['Date'].dt.month

train_size = len(df_monthly) - 12
train_df = df_monthly.iloc[:train_size]
test_df = df_monthly.iloc[train_size:]

X_train = train_df[['Time_Index', 'Month']]
y_train = train_df['Sales']

X_test = test_df[['Time_Index', 'Month']]
y_test = test_df['Sales']


model = LinearRegression()
model.fit(X_train, y_train)


test_predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, test_predictions)
rmse = np.sqrt(mean_squared_error(y_test, test_predictions))


metrics_html = f"""
<div style="background-color: #f4f6f9; padding: 15px; border-radius: 8px; border-left: 5px solid #2ca02c; color: #000000;">
    <h3 style="margin-top: 0; color: #000000;">🧪 Model Evaluation (Test Data: Last 12 Months)</h3>
    <p style="margin: 5px 0; font-size: 15px; color: #000000;">
        <strong style="color: #000000;">Mean Absolute Error (MAE):</strong> ${mae:,.2f}
        <i style="color: #333333;">(On average, our predictions were off by this much)</i>
    </p>
    <p style="margin: 0; font-size: 15px; color: #000000;">
        <strong style="color: #000000;">Root Mean Squared Error (RMSE):</strong> ${rmse:,.2f}
    </p>
</div>
<br>
"""
display(HTML(metrics_html))

last_date = df_monthly['Date'].max()
future_dates = pd.date_range(start=last_date + pd.DateOffset(months=1), periods=12, freq='MS')
future_time_index = np.arange(len(df_monthly), len(df_monthly) + 12)

future_X = pd.DataFrame({'Time_Index': future_time_index, 'Month': future_dates.month})
future_forecast = model.predict(future_X)


fig = go.Figure()

# Plot Training Data (Historical)
fig.add_trace(go.Scatter(x=train_df['Date'], y=train_df['Sales'],
                         mode='lines+markers', name='Historical Data (Train)',
                         line=dict(color='#1f77b4', width=2)))

# Plot Testing Data (Actual Last 12 Months Sales)
fig.add_trace(go.Scatter(x=test_df['Date'], y=test_df['Sales'],
                         mode='lines+markers', name='Actual Sales (Test)',
                         line=dict(color='#ff7f0e', width=2)))

# Plot Testing Data (Model's Prediction for Last 12 Months)
fig.add_trace(go.Scatter(x=test_df['Date'], y=test_predictions,
                         mode='lines', name='Model Prediction (Test)',
                         line=dict(color='#2ca02c', width=2, dash='dash')))

# Plot Future Forecast (Next 12 Months)
fig.add_trace(go.Scatter(x=future_dates, y=future_forecast,
                         mode='lines+markers', name='Future Forecast (Next 12 Months)',
                         line=dict(color='#d62728', width=3, dash='dot')))


fig.update_layout(
    title='🚀 Predictive Analytics: Sales Forecasting Model',
    xaxis_title='Date',
    yaxis_title='Total Monthly Sales ($)',
    template='plotly_white',
    hovermode='x unified',
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01, bgcolor="rgba(255, 255, 255, 0.8)"),
    height=600
)

fig.show(config={'displayModeBar': False})